In [1]:
import ast

In [2]:
import csv

In [3]:
import pandas as pd

In [4]:
import numpy as np

In [5]:
import json

In [6]:
cyr = ['а','б','в','г','д','е','ё','ж','з','и','й','к','л','м','н','о','п','р','с','т','у','ф','х','ц','ч','ш','щ','ъ','ы','ь','э','ю','я','ґ','є','ї','ђ','љ','њ','ћ','џ', 'ў','ъ']
lat = ['a','b','v','g','d','e','jo','zh','z','i','j','k','l','m','n','o','p','r','s','t','u','f','h','ts','ch','sh','sczs','','y','','e','ju','ja','g','e','j','dzh','l','n','ch','dzh', 'w','o']
lat2 = ['ch','cz','rz','sz', 'a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','qu','r','s','t','u','v','w','x', 'y','z','ą', 'ć','ę', 'ł','ń','ó','ś','ź','ż','á','č','ď','é','ě','í','ň','ř', 'š','ť','ú','ů','ý','ž','ä','ľ','ĺ', 'ô', 'ŕ','è']
cyr2 = ['х', 'ч', 'ж', 'ш',  'а','б','ц','д','е','ф','г','х','и','й','к','л','м','н','о','п','кв','р','с','т','у','в','в','кс','ы','з','ом','ч','ен','л','н','у','ш','ж','ж','а','ч','д','э','е','и','н','рж','ш','т','у','у','и','ж','э','л','лл','уо','рр','э']
langss = ['-', 'c', 'c', 'c', 'l', 'l', 'l', 'c', 'l']

# Transliterates text into proper script: 'c' - Cyrillic into Latinic, 'l' - conversely.
def transliterate(sss, l_kind):
  try:
    sss2 = sss.lower()

    if l_kind == 'c':
        l_from, l_to = cyr, lat
    elif l_kind == 'l':
        l_from, l_to = lat2, cyr2
    else:
        return ""
    for c, l in zip(l_from, l_to):
        try:
            sss2 = sss2.replace(c, l)
        except:
            pass
  except:
    pass
  return sss2

In [7]:
# 0.6 changes
phon_corr = {'о':['у'],
             'у':['о'],
             'е':['я', 'и', 'i', 'і'],
             'я':['е'],
             'д':['т'],
             'з':['с', 'ш'],
             'ж':['ш'],
             'к':['г'],
             'т':['д'],
             'с':['з', 'ш'],
             'ш':['ж'],
             'г':['к', 'х'],
             'и':['е'],
             'i': ['е'],
             'і': ['е'],
             'ч': ['ц'],
             'ц': ['ч'],
             'б': ['п'],
             'п': ['б'],
             'х': ['г'],
            }
# 0.5 changes
phon_corr_1 = {'ў':['в'],
               'в':['ў'],
               'й':['ј'],
               'ј':['й'],
               'и':['ј'],
               'ј':['и'],
               'њ':['н'],
               'н':['њ'],
               'љ':['л'],
               'л':['љ'],
               'ш':['щ'],
               'щ':['ш']
              }

# 0.2 changes
phon_corr_2 = {'ё':['е'],
               'е':['ё'],
               'i':['и', 'й'],
               'і': ['и', 'й'],
               'и':['i', 'ї', 'і', 'й'],
               'ї':['и', 'й'],
               'й':['i', 'і', 'ї', 'и'],
              }

# 0.3 changes
phon_corr_3 = {'о':['а'],
               'а':['о', 'я'],
               'я': ['а'],
               'ю': ['у'],
               'у': ['ю'],
               'е':['э'],         
               'ы':['и', 'i', 'і'],
               'и':['ы'],
               'i': ['ы'],
               'і': ['ы']
              }
# 1.2 changes: ћ, ђ and ъ with anything

def next_step_Levenshtein(dists, str1, str2, i, j):
    dists[i, j] = 1000
    v = min(dists[i-1, j-1], dists[i-1, j], dists[i, j-1])

    if str1[i-1] == 'ь' or str2[j-1] == 'ь':
        # "ь" в любой из строк считается за совпадение с чем угодно
        dists[i, j] = v  # Нулевая стоимость
        return
            
    if str1[i-1] == str2[j-1] and dists[i-1, j-1] <= dists[i-1, j] and dists[i-1, j-1] <= dists[i, j-1]:
        # print('+', str1[i-1: i+2], '|', str2[j-1: j+2], v)
        dists[i, j] = v
    else:
        # print('-', str1[i-1: i+2], '|', str2[j-1: j+2], v)
        if i < len(str1) - 1 and j < len(str2) - 1 and (str1[i-1], str1[i]) == (str2[j], str2[j-1]) and dists[i, j] > v + 0.5:
            # print("+")
            dists[i, j] = v + 0.5
        if i < len(str1) - 2 and j < len(str2) - 2 and (str1[i-2], str1[i-1]) == (str2[j-1], str2[j-2]) and dists[i, j] > v + 0.5:
            # print("++")
            dists[i, j] = v + 0.5
        if str1[i-1] in phon_corr.keys() and str2[j-1] in phon_corr[str1[i-1]] and dists[i, j] > v + 0.6:
            dists[i, j] = v + 0.6
        if str1[i-1] in phon_corr_1.keys() and str2[j-1] in phon_corr_1[str1[i-1]] and dists[i, j] > v + 0.5:
            dists[i, j] = v + 0.5
        if str1[i-1] in phon_corr_2.keys() and str2[j-1] in phon_corr_2[str1[i-1]] and dists[i, j] > v + 0.2:
            dists[i, j] = v + 0.2        
        if dists[i, j] > v + 1:
            dists[i, j] = v + 1

def find_path_Levenstein(dists, str1, str2):
    path = [] # [('>', '>', float(dists[len(str1)+1, len(str2)+1]))]
    pos1, pos2 = len(str1), len(str2)
    while pos1 > 0 and pos2 > 0:
        if dists[pos1, pos2] < dists[pos1, pos2+1] and dists[pos1, pos2] < dists[pos1+1, pos2]:
            path.append((str1[pos1-1], str2[pos2-1], float(dists[pos1, pos2])))
            pos1 -= 1
            pos2 -= 1
        elif dists[pos1+1, pos2] <= dists[pos1, pos2] and dists[pos1+1, pos2] <= dists[pos1+1, pos2]:
            path.append(('_', str2[pos2-1], float(dists[pos1, pos2])))
            pos2 -= 1
        else:
            path.append((str1[pos1-1], '_', float(dists[pos1, pos2])))
            pos1 -= 1

    if pos1 != 0:
        while pos1 != 0:
            path.append((str1[pos1-1], '_', float(dists[pos1, 0])))
            pos1 -= 1
    elif pos2 != 0:
        while pos2 != 0:
            path.append(('_', str2[pos2-1], float(dists[0, pos2])))
            pos2 -= 1
    return path

def fast_Levenstein(str1, str2):
    dists = np.zeros((len(str1)+2, len(str2)+2))
    dists[0, :] = np.arange(0, len(str2)+2, 1)
    dists[:, 0] = np.arange(0, len(str1)+2, 1)
    # print(dists)
    for i in range(1, len(str1)+1):
        for j in range(1, len(str2)+1):
            # print(i, j)
            # print(str1[i-1], str2[j-1])
            next_step_Levenshtein(dists, str1, str2, i, j)
            
        dists[i, len(str2)+1] = min(dists[i-1, len(str2)], dists[i-1, len(str2)+1], dists[i, len(str2)]) + 1
        # print(dists)
        
    for j in range(1, len(str2)+1):
        dists[len(str1)+1, j] = min(dists[len(str1), j-1], dists[len(str1), j], dists[len(str1)+1, j-1]) + 1
    dists[len(str1)+1, len(str2)+1] = dists[len(str1), len(str2)]
    # print(dists)
    path = find_path_Levenstein(dists, str1, str2)
    # print(path[::-1])
    return path[::-1][-1][-1]

In [8]:
russian_true = pd.read_csv('complete_morphs/russian_morphs.csv')
answers_true = pd.read_csv('complete_morphs/user_answers_morphs.csv')
ukrainian_true = pd.read_csv('complete_morphs/ukrainian_morphs.csv')
belarus_true = pd.read_csv('complete_morphs/belarus_morphs.csv')
bulgarian_true = pd.read_csv('complete_morphs/bulgarian_morphs.csv')
polish_true = pd.read_csv('complete_morphs/polish_morphs.csv')
czech_true = pd.read_csv('complete_morphs/czech_morphs.csv')
serbian_true = pd.read_csv('complete_morphs/serbian_morphs.csv')
slovak_true = pd.read_csv('complete_morphs/slovak_morphs.csv')
slovene_true = pd.read_csv('complete_morphs/slovene_morphs.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'complete_morphs/russian_morphs.csv'

In [ ]:
ukrainian_ans = answers_true.loc[answers_true['parallel_lang'] == 'Ukranian']
belarus_ans = answers_true.loc[answers_true['parallel_lang'] == 'Belarussian']
czech_ans = answers_true.loc[answers_true['parallel_lang'] == 'Czech']
polish_ans = answers_true.loc[answers_true['parallel_lang'] == 'Polish']
bulgarian_ans = answers_true.loc[answers_true['parallel_lang'] == 'Bulgarian']
serbian_ans = answers_true.loc[answers_true['parallel_lang'] == 'Serbian']
slovak_ans = answers_true.loc[answers_true['parallel_lang'] == 'Slovak']
slovene_ans = answers_true.loc[answers_true['parallel_lang'] == 'Slovene']
nopar_ans = answers_true.loc[answers_true['parallel_lang'] == 'No Parallel Text']

In [ ]:
num_type = {}

for _, row in nopar_ans.iterrows():
    test_id = row['test_id']
    answer_no = row['answer_no']
    correctness = row['correctness']
    code = (int(f'{test_id}{answer_no}'))
    if code not in num_type.keys():
        num_type[code] = {}
    if correctness not in num_type[code].keys():
        num_type[code][correctness] = 1
    else:
        num_type[code][correctness] +=1
for key in num_type:    
    sum_ans = 0
    for num in num_type[key]:
        sum_ans += num_type[key][num]
    if 'incorr' in num_type[key]:
        num_type[key]['mistakes'] = num_type[key]['incorr'] / sum_ans
    else:
        num_type[key]['mistakes'] = 0

for key in num_type:
    num_type[key] = dict(sorted(num_type[key].items(), key=lambda item: item[1], reverse=True))

num_type

In [ ]:
def cognate_searcher(answer_set, russian_true, language):
    result_dict = {}
    for _, row in answer_set.iterrows():
        test_id = row['test_id']
        answer_no = row['answer_no']
        user_answer = str(row['answer']).lower().strip()
        correctness = row['correctness']
        code_ans = (user_answer, correctness)
        
        correct_row = russian_true[
            (russian_true['test_id'] == test_id) & 
            (russian_true['answer_no'] == answer_no)
        ]
        original_row = language[
            (language['test_id'] == test_id) & 
            (language['answer_no'] == answer_no)
        ]
        
        if not correct_row.empty:
            correct_answer = str(correct_row.iloc[0]['answer']).strip()
            original_answer = str(original_row.iloc[0]['answer']).strip()
            code = (int(f'{test_id}{answer_no}'), correct_answer, original_answer)
            if code not in result_dict.keys():
                result_dict[code] = {}
        if code_ans not in result_dict[code].keys():
            result_dict[code][code_ans] = 1
        else:
            result_dict[code][code_ans] +=1
    
    for key in result_dict:
        result_dict[key] = dict(sorted(result_dict[key].items(), key=lambda item: item[1], reverse=True))

    return result_dict

In [ ]:
ukr_cognates = cognate_searcher(ukrainian_ans, russian_true, ukrainian_true)
ukr_cognates

In [ ]:
bel_cognates = cognate_searcher(belarus_ans, russian_true, belarus_true)
bel_cognates

In [ ]:
bg_cognates = cognate_searcher(bulgarian_ans, russian_true, bulgarian_true)
bg_cognates

In [ ]:
cz_cognates = cognate_searcher(czech_ans, russian_true, czech_true)
cz_cognates

In [ ]:
pol_cognates = cognate_searcher(polish_ans, russian_true, polish_true)
pol_cognates

In [ ]:
sn_cognates = cognate_searcher(slovene_ans, russian_true, slovene_true)
sn_cognates

In [ ]:
sk_cognates = cognate_searcher(slovak_ans, russian_true, slovak_true)
sk_cognates

In [ ]:
sb_cognates = cognate_searcher(serbian_ans, russian_true, serbian_true)
sb_cognates

In [ ]:
def morph_comparer(set_ru, set_cg, orpho):
    
    def compare_morphemes(ru_list, cg_list, morpheme_type):

        scores = []
        pairs = []

        if not ru_list and not cg_list:
            return {'scores': scores, 'pairs': pairs}

        if not ru_list:
            cg_combined = ''.join(cg_list)
            scores.append(fast_Levenstein('', cg_combined))
            pairs.append(('', cg_combined))
            return {'scores': scores, 'pairs': pairs}

        if not cg_list:
            ru_combined = ''.join(ru_list)
            scores.append(fast_Levenstein(ru_combined, ''))
            pairs.append((ru_combined, ''))
            return {'scores': scores, 'pairs': pairs}

        if len(ru_list) == len(cg_list):
            for ru_elem, cg_elem in zip(ru_list, cg_list):
                scores.append(fast_Levenstein(ru_elem, cg_elem))
                pairs.append((ru_elem, cg_elem))
            return {'scores': scores, 'pairs': pairs}

        if morpheme_type in ['SUFF', 'PREF']:
            return {'scores': ['HANDCHECK'], 'pairs': [ru_list, cg_list]}

    result = {}

    ru_dict = ast.literal_eval(set_ru)
    cg_dict = ast.literal_eval(set_cg)

    if orpho == 'latin':
        if isinstance(cg_dict, dict):
            for key in cg_dict:
                cg_dict[key] = [transliterate(el, 'l') for el in cg_dict[key]]

    morpheme_types = ['PREF', 'ROOT', 'LINK', 'SUFF', 'END', 'POSTFIX']
    not_link = 0

    for morpheme_type in morpheme_types:

        ru_list = []
        cg_list = []

        if isinstance(ru_dict, dict) and isinstance(cg_dict, dict):
            ru_list = ru_dict.get(morpheme_type, [])
            cg_list = cg_dict.get(morpheme_type, [])

        if not ru_list and not cg_list:
            continue

        if morpheme_type == 'ROOT' and len(ru_list) > len(cg_list) and len(cg_list) != 0:
            not_link = 1
            if 'LINK' in ru_dict.keys():
                ru_root = ru_list[0] + ru_dict['LINK'][0] + ru_list[1]
            else:
                ru_root = ru_list[0] + ru_list[1]
            result[morpheme_type] = {'scores': [fast_Levenstein(ru_root, cg_list[0])], 'pairs': [(ru_root, cg_list[0])]}
            continue

        if morpheme_type == 'ROOT' and len(cg_list) > len(ru_list) and len(ru_list) != 0:
            not_link = 1
            if 'LINK' in cg_dict.keys():
                cg_root = cg_list[0] + cg_dict['LINK'][0] + cg_list[1]
            else:
                cg_root = cg_list[0] + cg_list[1]
            result[morpheme_type] = {'scores': [fast_Levenstein(ru_list[0], cg_root)], 'pairs': [(ru_list[0], cg_root)]}
            continue

        if not not_link:
            result[morpheme_type] = compare_morphemes(ru_list, cg_list, morpheme_type)

    return result

In [ ]:
def cognate_stats(answer_set, russian_true, language, language_dict, orpho, num_type):
    true_cognates = {}
    part_cognates = {}
    for _, row in answer_set.iterrows():
        test_id = row['test_id']
        answer_no = row['answer_no']
        user_answer = str(row['answer']).lower().strip()
        correctness = row['correctness']
        code_ans = (user_answer, correctness)
        morphs_ans = row['morphemic_parse']

        correct_row = russian_true[
            (russian_true['test_id'] == test_id) &
            (russian_true['answer_no'] == answer_no)
            ]
        original_row = language[
            (language['test_id'] == test_id) &
            (language['answer_no'] == answer_no)
            ]

        if not correct_row.empty:
            correct_answer = str(correct_row.iloc[0]['answer']).strip()
            original_answer = str(original_row.iloc[0]['answer']).strip()
            correct_morph = str(correct_row.iloc[0]['morphemic_parse']).strip()
            original_morph = str(original_row.iloc[0]['morphemic_parse']).strip()
            code = (int(f'{test_id}{answer_no}'), correct_answer, original_answer)
            if float(original_row.iloc[0]['cognate']) != 0:
                if float(original_row.iloc[0]['cognate']) == 1:
                    if code not in true_cognates.keys():
                        true_cognates[code] = {}
                        true_cognates[code]['morph_comparison'] = morph_comparer(correct_morph, original_morph, orpho)
                        true_cognates[code]['overall_stats'] = {}
                        true_cognates[code]['word_stats'] = {}

                elif code not in part_cognates.keys():
                    part_cognates[code] = {}
                    part_cognates[code]['morph_comparison'] = morph_comparer(correct_morph, original_morph, orpho)
                    part_cognates[code]['overall_stats'] = {}
                    part_cognates[code]['word_stats'] = {}

        num_ans = 0
        for key in language_dict[code]:
            num_ans += language_dict[code][key]

        if float(original_row.iloc[0]['cognate']) == 1:
            if correctness not in true_cognates[code]['overall_stats'].keys():
                counter = 0
                for key in language_dict[code]:
                    if key[1] == correctness:
                        counter += language_dict[code][key]
                true_cognates[code]['overall_stats'][correctness] = counter / num_ans
                if correctness == 'incorr':
                    mistakes_control = num_type[code[0]]['mistakes']
                    if mistakes_control != 0:
                        explained_mistakes = (mistakes_control - true_cognates[code]['overall_stats'][
                            correctness]) / mistakes_control
                    else:
                        explained_mistakes = 0 - true_cognates[code]['overall_stats'][correctness]
                    true_cognates[code]['overall_stats']['explained_mistakes'] = explained_mistakes

            if code_ans not in true_cognates[code]['word_stats'].keys():
                true_cognates[code]['word_stats'][code_ans] = (
                morph_comparer(morphs_ans, original_morph, orpho), language_dict[code][code_ans] / num_ans)

        elif float(original_row.iloc[0]['cognate']) == 0.5:
            if correctness not in part_cognates[code]['overall_stats'].keys():
                counter = 0
                for key in language_dict[code]:
                    if key[1] == correctness:
                        counter += language_dict[code][key]
                part_cognates[code]['overall_stats'][correctness] = counter / num_ans
                if correctness == 'incorr':
                    mistakes_control = num_type[code[0]]['mistakes']
                    if mistakes_control != 0:
                        explained_mistakes = (mistakes_control - part_cognates[code]['overall_stats'][
                            correctness]) / mistakes_control
                    else:
                        explained_mistakes = 0 - part_cognates[code]['overall_stats'][correctness]
                    part_cognates[code]['overall_stats']['explained_mistakes'] = explained_mistakes
            if code_ans not in part_cognates[code]['word_stats'].keys():
                part_cognates[code]['word_stats'][code_ans] = (
                morph_comparer(morphs_ans, original_morph, orpho), language_dict[code][code_ans] / num_ans)

    for key in true_cognates:
        for el in true_cognates[key]:
            if el == 'overall_stats':
                true_cognates[key][el] = dict(
                    sorted(true_cognates[key][el].items(), key=lambda item: item[1], reverse=True))
            elif el == 'word_stats':
                true_cognates[key][el] = dict(
                    sorted(true_cognates[key][el].items(), key=lambda item: item[1][1], reverse=True))

    for key in part_cognates:
        for el in part_cognates[key]:
            if el == 'overall_stats':
                part_cognates[key][el] = dict(
                    sorted(part_cognates[key][el].items(), key=lambda item: item[1], reverse=True))
            elif el == 'word_stats':
                part_cognates[key][el] = dict(
                    sorted(part_cognates[key][el].items(), key=lambda item: item[1][1], reverse=True))

    return true_cognates, part_cognates

In [ ]:
ukr_stats = cognate_stats(ukrainian_ans, russian_true, ukrainian_true, ukr_cognates, 'cyr', num_type)
ukr_stats

In [ ]:
bel_stats = cognate_stats(belarus_ans, russian_true, belarus_true, bel_cognates, 'cyr', num_type)
bel_stats

In [ ]:
cz_stats = cognate_stats(czech_ans, russian_true, czech_true, cz_cognates, 'latin', num_type)
cz_stats

In [ ]:
pol_stats = cognate_stats(polish_ans, russian_true, polish_true, pol_cognates, 'latin', num_type)
pol_stats

In [ ]:
bg_stats = cognate_stats(bulgarian_ans, russian_true, bulgarian_true, bg_cognates, 'cyr', num_type)
bg_stats

In [ ]:
sb_stats = cognate_stats(serbian_ans, russian_true, serbian_true, sb_cognates, 'cyr', num_type)
sb_stats

In [ ]:
sk_stats = cognate_stats(slovak_ans, russian_true, slovak_true, sk_cognates, 'latin', num_type)
sk_stats

In [ ]:
sn_stats = cognate_stats(slovene_ans, russian_true, slovene_true, sn_cognates, 'latin', num_type)
sn_stats

In [ ]:
import pandas as pd
import numpy as np

def dict_to_dataframe(data):
    rows = []
    
    for key, value in data.items():
        row = {
            'pair': f"{key[0]}, '{key[1]}', '{key[2]}'"
        }
        
        overall = value.get('overall_stats', {})
        row['explained_mistakes'] = overall.get('explained_mistakes', 0.0)
        
        morph = value.get('morph_comparison', {})
        morph_categories = ['PREF', 'ROOT', 'LINK', 'SUFF', 'END', 'POSTFIX']
        
        for category in morph_categories:
            if category in morph:
                row[category] = morph[category].get('scores', [0.0])
            else:
                row[category] = [0.0]
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    columns_order = ['pair'] + morph_categories + ['explained_mistakes']
    df = df[columns_order]
    
    return df

ukr_true_df = dict_to_dataframe(ukr_stats[0])
ukr_true_df

In [ ]:
ukr_true_df.to_csv(
    'ukr_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [ ]:
bel_true_df = dict_to_dataframe(bel_stats[0])
bel_true_df.to_csv(
    'bel_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

cz_true_df = dict_to_dataframe(cz_stats[0])
cz_true_df.to_csv(
    'cz_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
pol_true_df = dict_to_dataframe(pol_stats[0])
pol_true_df.to_csv(
    'pol_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
bg_true_df = dict_to_dataframe(bg_stats[0])
bg_true_df.to_csv(
    'bg_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sn_true_df = dict_to_dataframe(sn_stats[0])
sn_true_df.to_csv(
    'sn_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sk_true_df = dict_to_dataframe(sk_stats[0])
sk_true_df.to_csv(
    'sk_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sb_true_df = dict_to_dataframe(sb_stats[0])
sb_true_df.to_csv(
    'sb_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [ ]:
import pandas as pd
import numpy as np

def alternate_to_dataframe(data):
    rows = []
    
    for key, value in data.items():
        code = key[0]
        original = key[2]
        overall = value.get('overall_stats', {})
        explained_mistakes = overall.get('explained_mistakes', 0.0)
        for key1, value1 in value['word_stats'].items():
            row = {
                'pair': f"{code}, '{key1[0]}', '{original}'"
            }
            row['explained_mistakes'] = explained_mistakes
            morph = value1[0]
            morph_categories = ['PREF', 'ROOT', 'LINK', 'SUFF', 'END', 'POSTFIX']
            
            for category in morph_categories:
                if category in morph:
                    row[category] = morph[category].get('scores', [0.0])
                else:
                    row[category] = [0.0]
            
            rows.append(row)
    
    df = pd.DataFrame(rows)
    
    columns_order = ['pair'] + morph_categories + ['explained_mistakes']
    df = df[columns_order]
    
    return df

ukr_part1_df = alternate_to_dataframe(ukr_stats[1])
ukr_part1_df

In [ ]:
ukr_part1_df.to_csv(
    'ukr_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

bel_part1_df = alternate_to_dataframe(bel_stats[1])
bel_part1_df.to_csv(
    'bel_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [ ]:
bg_part1_df = alternate_to_dataframe(bg_stats[1])
bg_part1_df.to_csv(
    'bg_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
pol_part1_df = alternate_to_dataframe(pol_stats[1])
pol_part1_df.to_csv(
    'pol_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
cz_part1_df = alternate_to_dataframe(cz_stats[1])
cz_part1_df.to_csv(
    'cz_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sb_part1_df = alternate_to_dataframe(sb_stats[1])
sb_part1_df.to_csv(
    'sb_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sn_part1_df = alternate_to_dataframe(sn_stats[1])
sn_part1_df.to_csv(
    'sn_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sk_part1_df = alternate_to_dataframe(sk_stats[1])
sk_part1_df.to_csv(
    'sk_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [ ]:
def transform_morpheme_dataframe(filepath):
    
    df = pd.read_csv(filepath)

    def safe_eval_list(val, default_len=3):
        if pd.isna(val) or val == '' or val == '[]':
            return [0] * default_len
        try:
            if isinstance(val, str):
                val = val.replace("'", '"')
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return parsed + [0] * (default_len - len(parsed))
                else:
                    return [parsed] + [0] * (default_len - 1)
            elif isinstance(val, list):
                return val + [0] * (default_len - len(val))
            else:
                return [val] + [0] * (default_len - 1)
        except:
            return [0] * default_len
    
    def safe_eval_single(val):
        if pd.isna(val) or val == '' or val == '[]':
            return 0
        try:
            if isinstance(val, str):
                val = val.replace("'", '"')
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return parsed[0] if parsed else 0
                else:
                    return parsed
            else:
                return val
        except:
            return 0
    
    new_data = []
    
    for idx, row in df.iterrows():
        new_row = {
            'pair': row['pair'],
            'explained_mistakes': row['explained_mistakes']
        }
        
        pref_list = safe_eval_list(row['PREF'], 2)
        new_row['PREF1'] = pref_list[0]
        new_row['PREF2'] = pref_list[1]
        
        root_list = safe_eval_list(row['ROOT'], 2)
        new_row['ROOT1'] = root_list[0]
        new_row['ROOT2'] = root_list[1]

        new_row['LINK'] = safe_eval_single(row['LINK'])
        
        suff_list = safe_eval_list(row['SUFF'], 3)
        new_row['SUFF1'] = suff_list[0]
        new_row['SUFF2'] = suff_list[1]
        new_row['SUFF3'] = suff_list[2]
        
        new_row['END'] = safe_eval_single(row['END'])
        
        new_row['POSTFIX'] = safe_eval_single(row['POSTFIX'])
        
        new_data.append(new_row)
    
    new_df = pd.DataFrame(new_data)
    
    columns_order = ['pair', 'PREF1', 'PREF2', 'ROOT1', 'ROOT2', 'LINK', 
                     'SUFF1', 'SUFF2', 'SUFF3', 'END', 'POSTFIX', 
                     'explained_mistakes']
    new_df = new_df[columns_order]
    
    return new_df